In [1]:
# Import required packages and pre-define some functions
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
nltk.download('vader_lexicon')
nltk.download('punkt')
import language_tool_python
grammar_check_tool = language_tool_python.LanguageTool('en-US')
from pysentimiento.preprocessing import preprocess_tweet
from pysentimiento import create_analyzer
emotion_analyzer = create_analyzer(task="emotion", lang="en")
hate_speech_analyzer = create_analyzer(task="hate_speech", lang="en")
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from scipy.special import softmax
import urllib.request
from scipy.special import expit
import time

sia = SentimentIntensityAnalyzer()
# def assign_sentiment(x):
    # if x >= 0.05:
    #     return 'positive'
    # elif x <= -0.05:
    #     return 'negative'
    # else:
    #     return 'neutral'
def assign_sentiment(neg, pos):
    if neg > pos:
        label = "negative"
    elif pos > neg:
        label = "positive"
    else:
        label = "neutral"
    return label

# Preprocess text for irony measurement (username and link placeholders)
def roberta_preprocess(irony_text):
    new_text = [
    ]
    for t in irony_text.split(" "):
        t = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)

# Initialize the irony model
task_irony='irony'
MODEL_irony = f"cardiffnlp/twitter-roberta-base-{task_irony}"
tokenizer_irony = AutoTokenizer.from_pretrained(MODEL_irony)
model_irony = AutoModelForSequenceClassification.from_pretrained(MODEL_irony)
# model_irony.save_pretrained(MODEL_irony)

task_offensive='offensive'
MODEL_offensive = f"cardiffnlp/twitter-roberta-base-{task_offensive}"
tokenizer_offensive = AutoTokenizer.from_pretrained(MODEL_offensive)
model_offensive = AutoModelForSequenceClassification.from_pretrained(MODEL_offensive)
# model_offensive.save_pretrained(MODEL_offensive)

task_emoji='emoji'
MODEL_emoji = f"cardiffnlp/twitter-roberta-base-{task_emoji}"
tokenizer_emoji = AutoTokenizer.from_pretrained(MODEL_emoji)
model_emoji = AutoModelForSequenceClassification.from_pretrained(MODEL_emoji)
# model_emoji.save_pretrained(MODEL_emoji)

# Readability model
import readability
import syntok.segmenter as segmenter
def read_tokenized(text):
    tokenized = '\n\n'.join('\n'.join(' '.join(token.value for token in sentence) for sentence in paragraph) for paragraph in segmenter.analyze(text))
    return tokenized

# Topic model-multi
MODEL_topic = f"cardiffnlp/tweet-topic-21-multi"
tokenizer_topic = AutoTokenizer.from_pretrained(MODEL_topic)
model_topic = AutoModelForSequenceClassification.from_pretrained(MODEL_topic)
class_mapping = model_topic.config.id2label

# Topic model-single
MODEL_topic_single = f"cardiffnlp/tweet-topic-21-single"
tokenizer_topic_single = AutoTokenizer.from_pretrained(MODEL_topic_single)
model_topic_single = AutoModelForSequenceClassification.from_pretrained(MODEL_topic_single)
class_mapping_single = model_topic_single.config.id2label

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/jonasbjaerke/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/jonasbjaerke/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [2]:
from spacy.lang.en.stop_words import stopword
import re
nltk.download('vader_lexicon')
nltk.download('punkt')
nltk.download('stopwords')
from wordcloud import WordCloud, STOPWORDS
from nltk.stem import SnowballStemmer
stop_words = nltk.corpus.stopwords.words('english')
import string
import preprocessor as p

def berkem_preprocess(text):
    text = re.sub('RT @\w+: '," ",text)
    text = re.sub("(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)"," ", text)
    p.set_options(p.OPT.URL, p.OPT.EMOJI, p.OPT.MENTION, p.OPT.NUMBER)
    text = p.clean(text)
    text = re.sub(r'#', "", text)
    text = re.sub(r'&amp', "", text)
    text = text.lower()

    return text

def grammar_p(text):
    text = re.sub('RT @\w+: '," ",text)
    text = re.sub("(@[A-Za-z0-9]+)|([^0-9A-Za-z \t])|(\w+:\/\/\S+)"," ", text)
    text = p.clean(text)
    return text

def get_grammar_score(text):
    scores_word_based_sentence = []
    scores_sentence_based_sentence = []
    # s1 = time.perf_counter()
    sentences = nltk.tokenize.sent_tokenize(text)
    # e1 = time.perf_counter()
    # sentences = self.split_into_sentences(row)
    for sentence in sentences:
    # for sentence in helpers.text_to_sentences(text):
        matches = grammar_check_tool.check(sentence)
        count_errors = len(matches)
        # only check if the sentence is correct or not
        scores_sentence_based_sentence.append(np.min([count_errors, 1]))
        scores_word_based_sentence.append(count_errors)

    word_count = len(nltk.tokenize.word_tokenize(text))
    sum_count_errors_word_based = np.sum(scores_word_based_sentence)
    score_word_based = 1 - (sum_count_errors_word_based / word_count)
    sentence_count = len(sentences)
    sum_count_errors_sentence_based = np.sum(scores_sentence_based_sentence)
    score_sentence_based = 1 - np.sum(sum_count_errors_sentence_based / sentence_count)

    return [score_word_based, score_sentence_based]

def clean_text(text):
    text_lc = "".join([word.lower() for word in text if word not in string.punctuation]) # remove puntuation
    text_rc = re.sub('[0-9]+', '', text_lc)
    tokens = re.split('\W+', text_rc)    # tokenization
    stemmer = nltk.PorterStemmer()
    text = [stemmer.stem(word) for word in tokens if word not in stopword]  # remove stopwords and stemming
    text = " ".join(text)
    return text

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /Users/jonasbjaerke/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/jonasbjaerke/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/jonasbjaerke/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import nltk
import numpy as np
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, recall_score, precision_score
import json

def text_preprocessing(text):
    tokens = nltk.word_tokenize(text)
    stemmer = nltk.stem.porter.PorterStemmer()
    roots = [stemmer.stem(tok) for tok in tokens]
    result = " ".join(roots)
    return result

In [ ]:
# twigen = pd.read_csv("gender-classifier-DFE-791531.csv", encoding='latin1')
# print(twigen.iloc[:5])

# # concat gender and description
# data = pd.concat([twigen["gender"], twigen["description"]], axis=1)

# # drop nan values
# data.dropna(inplace=True, axis=0)

# # convert genders from female and male to 1 and 0 respectively
# mapping_gender = {'female': 0, 'male': 1, "brand": 2, "unknown": 3}
# data["gender"] = data['gender'].map(mapping_gender)
# data = data[data["gender"].isin([0, 1])]
# data["description"] = data["description"].apply(lambda x: text_preprocessing(x))

# vectorizer = TfidfVectorizer()
# X = vectorizer.fit_transform(data["description"])
# y = data['gender']

# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# model = LogisticRegression()
# model.fit(X_train, y_train)

# y_pred = model.predict(X_test)

# accuracy = accuracy_score(y_test, y_pred)
# print("Accuracy: %.2f%%" % (accuracy * 100.0))

# recall = recall_score(y_test, y_pred)
# print("Recall: %.2f" % recall)

# precision = precision_score(y_test, y_pred)
# print("Precision: %.2f" % precision)

# f1 = f1_score(y_test, y_pred)
# print("F1 Score: %.2f" % f1)

# conf_matrix = confusion_matrix(y_test, y_pred)
# print("Confusion Matrix:")
# print(conf_matrix)

In [4]:
hashtags = [
    "#Trump"]
# hashtags = ["#Avengers"]

tweets_dic = {}

path_ms = "test.csv"
num_tweets, num_users = 0, 0
for h in hashtags:
    filename1 = path_ms
    tweets_dic = pd.read_csv(filename1, dtype=str, engine="python")

    print(f"{h} has {len(tweets_dic)} tweets")
    num_tweets += len(tweets_dic)
print(f"these hashtags have totally {num_tweets} tweets.")


#Trump has 8915 tweets
these hashtags have totally 8915 tweets.


In [5]:
def m_mapping_category_values(df):
    # map category values to numeric values
    df_copy = df.copy()
    mapping_sentiment_overall = {'neutral': 0, 'negative': 1, 'positive': 2}
    df_copy['sentiment_overall'] = df_copy['sentiment_overall'].map(mapping_sentiment_overall)

    mapping_emo_overall = {'others': 0, 'joy': 1, 'sadness': 2, 'disgust': 3, 'surprise': 4, 'anger': 5, 'fear': 6}
    df_copy['emo_overall'] = df_copy['emo_overall'].map(mapping_emo_overall)

    mapping_topic_overall = {'arts_&_culture': 0, 'business_&_entrepreneurs': 1, 'celebrity_&_pop_culture': 2, 'diaries_&_daily_life': 3, 'family': 4, 'fashion_&_style': 5, 'film_tv_&_video': 6,
                          'fitness_&_health': 7, 'food_&_dining': 8, 'gaming': 9, 'learning_&_educational': 10, 'music': 11, 'news_&_social_concern': 12, 'other_hobbies': 13,
                          'relationships': 14, 'science_&_technology': 15, 'sports': 16, 'travel_&_adventure': 17, 'youth_&_student_life': 18}
    df_copy['topic_overall'] = df_copy['topic_overall'].map(mapping_topic_overall)

    mapping_single_topic_overall = {'arts_&_culture': 0, 'business_&_entrepreneurs': 1, 'pop_culture': 2, 'daily_life': 3, 'sports_&_gaming': 4, 'science_&_technology': 5}
    df_copy['single_topic_overall'] = df_copy['single_topic_overall'].map(mapping_single_topic_overall)

    return df_copy

In [6]:
import time
import numpy as np
import readability

# Calculate features for one tweet table (not split by hashtag)

start_time = time.perf_counter()

ms_raw_data = tweets_dic.copy()
ms_raw_data = ms_raw_data.dropna(subset=["full_text"]).copy()

unique_text = ms_raw_data["full_text"].astype(str).unique()

# There are too many repeated texts wasting a lot of computing resources and time,
# so only measure each unique text once and store the results in dictionaries.
text_len, word_count = {}, {}
sentiment_scores = {}
matches, grammar_score = {}, {}
subjectivity_score, polarity_score = {}, {}
emo_results, hs_results = {}, {}
irony_scores, offensive_scores = {}, {}
emoji_count = {}
readability_results = {}
topic_scores, topic_pred = {}, {}
topic_scores_single, topic_pred_single = {}, {}

e_count = 0
remove_list = []

def log_processing_error(step, idx, text, e):
    print(f"[ERROR {step}] text #{idx}: {repr(str(text)[:200])}")
    print(f"  error: {type(e).__name__}: {e}")

for idx, text in enumerate(unique_text, 1):
    text = str(text)

    try:
        grammar_text = grammar_p(text)
        berkem_text = berkem_preprocess(text)
    except Exception as e:
        log_processing_error("preprocess", idx, text, e)
        e_count += 1
        remove_list.append(text)
        continue

    # Length of text and number of words
    try:
        text_len[text] = len(berkem_text)
        word_count[text] = len(str(berkem_text).split())
    except Exception as e:
        log_processing_error("length/word_count", idx, text, e)
        e_count += 1
        remove_list.append(text)
        continue

    # Sentiment
    try:
        sentiment_scores[text] = sia.polarity_scores(berkem_text)
    except Exception as e:
        log_processing_error("sentiment", idx, text, e)
        e_count += 1
        remove_list.append(text)
        continue

    # Grammar
    try:
        matches[text] = grammar_check_tool.check(text)
        grammar_score[text] = get_grammar_score(grammar_text)
    except Exception as e:
        log_processing_error("grammar", idx, text, e)
        e_count += 1
        remove_list.append(text)
        continue

    # Language scores
    try:
        subjectivity_score[text] = TextBlob(text).sentiment[1]
        polarity_score[text] = TextBlob(text).sentiment[0]
    except Exception as e:
        log_processing_error("textblob", idx, text, e)
        e_count += 1
        remove_list.append(text)
        continue

    # Emotion
    try:
        emo_results[text] = emotion_analyzer.predict(preprocess_tweet(text))
    except Exception as e:
        log_processing_error("emotion", idx, text, e)
        e_count += 1
        remove_list.append(text)
        continue

    # Hate speech
    try:
        hs_results[text] = hate_speech_analyzer.predict(preprocess_tweet(text))
    except Exception as e:
        log_processing_error("hate_speech", idx, text, e)
        e_count += 1
        remove_list.append(text)
        continue

    # Irony
    try:
        encoded_input_irony = tokenizer_irony(
            roberta_preprocess(text),
            return_tensors="pt"
        )
        output_irony = model_irony(**encoded_input_irony)
        irony_scores[text] = softmax(output_irony[0][0].detach().numpy())
    except Exception as e:
        log_processing_error("irony", idx, text, e)
        e_count += 1
        remove_list.append(text)
        continue

    # Offensive
    try:
        encoded_input_offensive = tokenizer_offensive(
            roberta_preprocess(text),
            return_tensors="pt"
        )
        output_offensive = model_offensive(**encoded_input_offensive)
        offensive_scores[text] = softmax(output_offensive[0][0].detach().numpy())
    except Exception as e:
        log_processing_error("offensive", idx, text, e)
        e_count += 1
        remove_list.append(text)
        continue

    # Emoji
    try:
        encoded_input_emoji = tokenizer_emoji(
            roberta_preprocess(text),
            return_tensors="pt"
        )
        output_emoji = model_emoji(**encoded_input_emoji)
        emoji_scores_local = softmax(output_emoji[0][0].detach().numpy())
        ranking = np.argsort(emoji_scores_local)[::-1]
        emoji_count[text] = ranking[0]
    except Exception as e:
        log_processing_error("emoji", idx, text, e)
        e_count += 1
        remove_list.append(text)
        continue

    # Readability
    try:
        # readability.getmeasures expects tokenized text:
        # tokens separated by spaces, one sentence per line
        readability_results[text] = readability.getmeasures(
            read_tokenized(berkem_text),
            lang="en"
        )
    except Exception as e:
        log_processing_error("readability", idx, text, e)
        e_count += 1
        readability_results[text] = None

    # Topic-multi
    try:
        tokens_topic = tokenizer_topic(text, return_tensors="pt")
        output_topic = model_topic(**tokens_topic)
        topic_scores[text] = expit(output_topic[0][0].detach().numpy())
        topic_pred[text] = (topic_scores[text] >= 0.5) * 1
    except Exception as e:
        log_processing_error("topic-multi", idx, text, e)
        e_count += 1
        topic_scores[text] = None
        topic_pred[text] = None
        remove_list.append(text)

    # Topic-single
    try:
        tokens_topic_single = tokenizer_topic_single(
            roberta_preprocess(text),
            return_tensors="pt"
        )
        output_topic_single = model_topic_single(**tokens_topic_single)
        topic_scores_single[text] = expit(output_topic_single[0][0].detach().numpy())
        topic_pred_single[text] = (topic_scores_single[text] >= 0.5) * 1
    except Exception as e:
        log_processing_error("topic-single", idx, text, e)
        e_count += 1
        topic_scores_single[text] = None
        topic_pred_single[text] = None
        remove_list.append(text)

# Remove rows whose text failed any required step
ms_raw_data = ms_raw_data[
    ~ms_raw_data["full_text"].astype(str).isin(remove_list)
].copy()

# Length of text and number of words
ms_raw_data["text_len"] = ms_raw_data["full_text"].apply(lambda x: text_len[str(x)])
ms_raw_data["word_count"] = ms_raw_data["full_text"].apply(lambda x: word_count[str(x)])

# Sentiment scores
ms_raw_data["neg"] = ms_raw_data["full_text"].apply(lambda x: sentiment_scores[str(x)]["neg"])
ms_raw_data["neu"] = ms_raw_data["full_text"].apply(lambda x: sentiment_scores[str(x)]["neu"])
ms_raw_data["pos"] = ms_raw_data["full_text"].apply(lambda x: sentiment_scores[str(x)]["pos"])
ms_raw_data["compound"] = ms_raw_data["full_text"].apply(lambda x: sentiment_scores[str(x)]["compound"])
ms_raw_data["sentiment_overall"] = ms_raw_data["full_text"].apply(
    lambda x: assign_sentiment(
        sentiment_scores[str(x)]["neg"],
        sentiment_scores[str(x)]["pos"]
    )
)

# Grammar
ms_raw_data["grammar-word-score"] = ms_raw_data["full_text"].apply(lambda x: grammar_score[str(x)][0])
ms_raw_data["grammar-sentence-score"] = ms_raw_data["full_text"].apply(lambda x: grammar_score[str(x)][1])

# Language
ms_raw_data["subjectivity"] = ms_raw_data["full_text"].apply(lambda x: subjectivity_score[str(x)])
ms_raw_data["polarity"] = ms_raw_data["full_text"].apply(lambda x: polarity_score[str(x)])

# Emotion
ms_raw_data["emo_overall"] = ms_raw_data["full_text"].apply(lambda x: emo_results[str(x)].output)
ms_raw_data["emo_anger"] = ms_raw_data["full_text"].apply(lambda x: emo_results[str(x)].probas["anger"])
ms_raw_data["emo_joy"] = ms_raw_data["full_text"].apply(lambda x: emo_results[str(x)].probas["joy"])
ms_raw_data["emo_fear"] = ms_raw_data["full_text"].apply(lambda x: emo_results[str(x)].probas["fear"])
ms_raw_data["emo_disgust"] = ms_raw_data["full_text"].apply(lambda x: emo_results[str(x)].probas["disgust"])
ms_raw_data["emo_surprise"] = ms_raw_data["full_text"].apply(lambda x: emo_results[str(x)].probas["surprise"])
ms_raw_data["emo_sadness"] = ms_raw_data["full_text"].apply(lambda x: emo_results[str(x)].probas["sadness"])
ms_raw_data["emo_others"] = ms_raw_data["full_text"].apply(lambda x: emo_results[str(x)].probas["others"])

# Hate speech
ms_raw_data["hs_aggressive"] = ms_raw_data["full_text"].apply(lambda x: hs_results[str(x)].probas["aggressive"])
ms_raw_data["hs_hateful"] = ms_raw_data["full_text"].apply(lambda x: hs_results[str(x)].probas["hateful"])
ms_raw_data["hs_targeted"] = ms_raw_data["full_text"].apply(lambda x: hs_results[str(x)].probas["targeted"])
ms_raw_data["hs_count"] = ms_raw_data["full_text"].apply(lambda x: len(hs_results[str(x)].output))

# Irony and offensive
ms_raw_data["irony"] = ms_raw_data["full_text"].apply(
    lambda x: 0 if irony_scores[str(x)][1] < 0.5 else 1
)
ms_raw_data["offensive"] = ms_raw_data["full_text"].apply(
    lambda x: offensive_scores[str(x)][1]
)

# Emoji
ms_raw_data["emoji"] = ms_raw_data["full_text"].apply(lambda x: emoji_count[str(x)])

# Readability
ms_raw_data["Kincaid"] = ms_raw_data["full_text"].apply(
    lambda x: np.nan if readability_results[str(x)] is None
    else readability_results[str(x)]["readability grades"]["Kincaid"]
)
ms_raw_data["ARI"] = ms_raw_data["full_text"].apply(
    lambda x: np.nan if readability_results[str(x)] is None
    else readability_results[str(x)]["readability grades"]["ARI"]
)
ms_raw_data["Coleman-Liau"] = ms_raw_data["full_text"].apply(
    lambda x: np.nan if readability_results[str(x)] is None
    else readability_results[str(x)]["readability grades"]["Coleman-Liau"]
)
ms_raw_data["FleschReadingEase"] = ms_raw_data["full_text"].apply(
    lambda x: np.nan if readability_results[str(x)] is None
    else readability_results[str(x)]["readability grades"]["FleschReadingEase"]
)
ms_raw_data["GunningFogIndex"] = ms_raw_data["full_text"].apply(
    lambda x: np.nan if readability_results[str(x)] is None
    else readability_results[str(x)]["readability grades"]["GunningFogIndex"]
)
ms_raw_data["LIX"] = ms_raw_data["full_text"].apply(
    lambda x: np.nan if readability_results[str(x)] is None
    else readability_results[str(x)]["readability grades"]["LIX"]
)
ms_raw_data["SMOGIndex"] = ms_raw_data["full_text"].apply(
    lambda x: np.nan if readability_results[str(x)] is None
    else readability_results[str(x)]["readability grades"]["SMOGIndex"]
)
ms_raw_data["RIX"] = ms_raw_data["full_text"].apply(
    lambda x: np.nan if readability_results[str(x)] is None
    else readability_results[str(x)]["readability grades"]["RIX"]
)
ms_raw_data["DaleChallIndex"] = ms_raw_data["full_text"].apply(
    lambda x: np.nan if readability_results[str(x)] is None
    else readability_results[str(x)]["readability grades"]["DaleChallIndex"]
)
ms_raw_data["complex_words"] = ms_raw_data["full_text"].apply(
    lambda x: np.nan if readability_results[str(x)] is None
    else readability_results[str(x)]["sentence info"]["complex_words"]
)
ms_raw_data["complex_words_dc"] = ms_raw_data["full_text"].apply(
    lambda x: np.nan if readability_results[str(x)] is None
    else readability_results[str(x)]["sentence info"]["complex_words_dc"]
)

# Topic-multi
for i in range(len(class_mapping)):
    ms_raw_data[class_mapping[i]] = ms_raw_data["full_text"].apply(
        lambda x: topic_scores[str(x)][i]
    )

ms_raw_data["topic_count"] = ms_raw_data["full_text"].apply(
    lambda x: sum(topic_pred[str(x)])
)
ms_raw_data["topic_overall"] = ms_raw_data["full_text"].apply(
    lambda x: class_mapping[
        topic_scores[str(x)].tolist().index(max(topic_scores[str(x)]))
    ]
)

# Topic-single
for i in range(len(class_mapping_single)):
    ms_raw_data["single_" + class_mapping_single[i]] = ms_raw_data["full_text"].apply(
        lambda x: topic_scores_single[str(x)][i]
    )

ms_raw_data["single_topic_count"] = ms_raw_data["full_text"].apply(
    lambda x: sum(topic_pred_single[str(x)])
)
ms_raw_data["single_topic_overall"] = ms_raw_data["full_text"].apply(
    lambda x: class_mapping_single[
        topic_scores_single[str(x)].tolist().index(max(topic_scores_single[str(x)]))
    ]
)

ms_raw_data = m_mapping_category_values(ms_raw_data)

finish_time = time.perf_counter()

output_file = "M_features.csv"
ms_raw_data.to_csv(output_file, encoding="utf-8", index=False)

print(f"saved {len(ms_raw_data)} rows to {output_file}")
print(f"removed {len(set(remove_list))} unique texts because of processing errors")
print(f"total processing errors: {e_count}")
print(f"finished in {(finish_time - start_time) / 60} minutes")

saved 8915 rows to M_features.csv
removed 0 unique texts because of processing errors
total processing errors: 0
finished in 31.102033829167098 minutes


In [10]:
import csv
import json
import math

def parse_value(value):
    if value is None:
        return None

    value = value.strip()

    if value == "":
        return None

    lower = value.lower()
    if lower == "nan":
        return math.nan
    if lower == "true":
        return True
    if lower == "false":
        return False

    try:
        num = float(value)
        if num.is_integer():
            return int(num)
        return num
    except ValueError:
        return value


def csv_to_json_dict(csv_path, json_path, key_field="post_uri", drop_fields=("full_text",)):
    result = {}

    with open(csv_path, "r", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            key = row.pop(key_field)

            for field in drop_fields:
                row.pop(field, None)

            result[key] = {k: parse_value(v) for k, v in row.items()}

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2, allow_nan=True)

    return result
data = csv_to_json_dict("M_features.csv", "output.json")